In [1]:
"""
Step 4 (Colab version): Fine-tune DistilBERT with LoRA (PEFT)
-------------------------------------------------------------------
HOW TO USE:
1. Go to https://colab.research.google.com -> New Notebook
2. Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4) -> Save
3. Paste Cell 1 into its own cell and RUN IT
4. Runtime -> Restart session (REQUIRED - do not skip this)
5. Paste Cells 2, 3, 4 into separate cells below, then run them in order
6. When prompted in Cell 2, upload your reviews_labeled.csv from your local data/ folder
7. Cell 4 zips and downloads the trained adapter automatically
"""

# ============================================================================
# CELL 1: Install dependencies (run this FIRST, then Restart session before
# continuing to Cell 2 - do not skip the restart, it's required for the
# torchao/torchvision fixes to actually take effect)
# ============================================================================
!pip install -q transformers datasets peft accelerate
!pip install -q -U torchao
!pip uninstall -y torchvision -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.0 MB/s eta 0:00:00


In [2]:
# CELL 2: Upload your reviews_labeled.csv
# (only run this AFTER restarting the session following Cell 1)
# ============================================================================
from google.colab import files
print("Upload your reviews_labeled.csv file:")
uploaded = files.upload()  # a file picker will pop up in the Colab UI

Upload your reviews_labeled.csv file:


Saving reviews_labeled.csv to reviews_labeled.csv


In [3]:
# CELL 3: Training
# ============================================================================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

INPUT_CSV = "reviews_labeled.csv"   # uploaded directly into Colab's working dir
BASE_MODEL = "distilbert-base-uncased"
OUTPUT_DIR = "/content/lora_sentiment_model"

LABEL2ID = {"negative": 0, "positive": 1}  # binary - Kaggle IMDB dataset has no neutral class
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("CUDA available:", torch.cuda.is_available())  # should print True on a GPU runtime

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

df = pd.read_csv(INPUT_CSV)
df = df.dropna(subset=["review_text", "sentiment"])
df["label"] = df["sentiment"].map(LABEL2ID)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)

train_ds = Dataset.from_pandas(train_df[["review_text", "label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["review_text", "label"]].reset_index(drop=True))

print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["review_text"], truncation=True, padding="max_length", max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["review_text"])
test_ds = test_ds.remove_columns(["review_text"])

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="/content/lora_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
    fp16=True,  # speeds up training significantly on a T4 GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

print("\nFinal evaluation:")
metrics = trainer.evaluate()
print(metrics)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nSaved LoRA adapter + tokenizer to {OUTPUT_DIR}")

import os
adapter_file = os.path.join(OUTPUT_DIR, "adapter_model.safetensors")
if os.path.exists(adapter_file):
    size_mb = os.path.getsize(adapter_file) / (1024 * 1024)
    print(f"Adapter size: {size_mb:.2f} MB (compare this to the ~260MB base model)")

CUDA available: True
Loading tokenizer and base model...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.257126,0.244901,0.900400,0.900399
2,0.232949,0.222105,0.909700,0.909693
3,0.184827,0.236328,0.910200,0.910159
4,0.185088,0.229438,0.913400,0.913397



Final evaluation:


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.185088,0.229438,4,0.913400,0.913397


{'eval_loss': 0.22943750023841858, 'eval_accuracy': 0.9134, 'eval_f1_macro': 0.9133972841388306}

Saved LoRA adapter + tokenizer to /content/lora_sentiment_model
Adapter size: 2.82 MB (compare this to the ~260MB base model)


In [4]:
# ============================================================================
# CELL 4: Zip and download the trained adapter back to your machine
# ============================================================================
!zip -r lora_sentiment_model.zip /content/lora_sentiment_model
files.download("lora_sentiment_model.zip")

  adding: content/lora_sentiment_model/ (stored 0%)
  adding: content/lora_sentiment_model/adapter_config.json (deflated 59%)
  adding: content/lora_sentiment_model/tokenizer.json (deflated 71%)
  adding: content/lora_sentiment_model/adapter_model.safetensors (deflated 7%)
  adding: content/lora_sentiment_model/README.md (deflated 66%)
  adding: content/lora_sentiment_model/tokenizer_config.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>